# 3. Causal features and evidence channels

Inspect the exact feature function used by development and inference.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))
from telco_anomaly.pipeline import runtime

POLICY = runtime(ROOT / "configs/pipeline.yml")
PACK = Path(POLICY["pack"])
RUN = Path(POLICY["run"])


In [ ]:
import yaml
from telco_anomaly.pipeline import observations, featurize
from telco_anomaly.features import fit_seasonality
from telco_anomaly.synthetic_pipeline import split_boundaries

grid, manifest = observations(POLICY)
experiment = yaml.safe_load(Path(POLICY["experiment"]).read_text())
train_end = split_boundaries(manifest)["train"][1]
decisions, _ = fit_seasonality(grid, train_end, POLICY["timezone"])
features = featurize(grid, manifest, experiment, decisions)
display(features.head())
display(features.select_dtypes("number").describe().T)
print("Feature columns:", [c for c in features if "__" in c])

Features include past-only robust deviations, interval FEC rates and nonzero indicators, directional power changes/asymmetry, and approved training-fitted daily residuals. Isolation Forest uses residual features, not raw levels or identifiers. Evidence channels add restart-aware drift, contemporaneous peer/common-mode changes, group silence and inventory-based optical margin. Missing history causes abstention. The static received-power comparator and the combined portfolio use the same incident rules.